## Data Loader

Necessary Imports

In [1]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
class LLM_Dataset_v1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids=[]
        self.target_ids=[]

        #tokenizing the entire text
        token_ids=tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        #sliding window to prevent LLM from seeing future tokens
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

        def __len__(self):
            return len(self.input_ids)
        
        def __getitem__(self, idx):
            return self.input_ids[idx], self.target_ids[idx]

In [3]:
def create_dataloader_v1(txt, batch_size, max_length, stride, shuffle=True, drop_last=True, num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=LLM_Dataset_v1(txt, tokenizer, max_length, stride)

    dataloader=DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers
    )
    return dataloader

## Multi Head attention class with causal masking

### self implementation

In [4]:
import torch
import torch.nn as nn

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out%num_heads==0, "output dimension must be divisible by number of heads"

        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in=x.shape

        keys=self.W_key(x)
        queries=self.W_query(x)
        values=self.W_value(x)

        #splitting the matrix
        keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values=values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys=keys.transpose(1,2)
        queries=queries.transpose(1,2)
        values=values.transpose(1,2)

        #computing self attention with a causal mask
        attn_scores=queries@keys.transpose(2,3)

        mask_bool=self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        attn_weights=self.dropout(attn_weights)

        context_vec=(attn_weights@values).transpose(1,2)

        context_vec=context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec=self.out_proj(context_vec)

        return context_vec

## GPT Transformer Architecture

In [6]:
import torch
import matplotlib
import tiktoken

In [7]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [8]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        #scale and shift are additional trainiable parameters just in case data processing scaling and shifting improves performance
        self.eps=1e-5 #handles division by zero just in case
        self.scale=nn.Parameter(torch.ones(emb_dim))
        self.shift=nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean=x.mean(dim=-1, keepdim=True)
        var=x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x=(x-mean)/torch.sqrt(var+self.eps)
        return self.scale*norm_x+self.shift

In [9]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.44715*torch.pow(x, 3))))

In [10]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [11]:
print(GPT_CONFIG_124M["emb_dim"])

768


In [12]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [13]:
print(GPT_CONFIG_124M)

{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}


In [14]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [15]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [16]:
torch.manual_seed(123)
model_small = GPTModel(GPT_CONFIG_124M)

out = model_small(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.0967, -0.0039, -0.2247,  ..., -0.0418, -0.1078,  0.1488],
         [ 0.3598, -0.8211, -0.6984,  ..., -0.5151,  0.2315, -0.3782],
         [ 0.6741, -0.1481, -0.1372,  ...,  0.1201, -0.6477, -0.0185],
         [-0.4303,  0.1975, -0.1071,  ...,  1.1665,  0.5099, -0.5637]],

        [[ 0.0709, -0.2920, -0.1572,  ..., -0.0820,  0.2679, -0.2921],
         [ 0.0675, -0.3491, -0.3243,  ...,  1.2713,  0.0092,  0.1581],
         [ 0.6107,  0.4507, -0.3838,  ...,  0.7442,  0.3833, -0.2200],
         [-0.1378, -0.0068,  0.4739,  ...,  1.2238, -0.4154,  0.0349]]],
       grad_fn=<UnsafeViewBackward0>)


In [17]:
total_params = sum(p.numel() for p in model_small.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


In [18]:
# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes = total_params * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


In [19]:
GPT_CONFIG_MEDIUM = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 1024,         # Embedding dimension
    "n_heads": 16,          # Number of attention heads
    "n_layers": 24,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [20]:
torch.manual_seed(123)
model_medium = GPTModel(GPT_CONFIG_MEDIUM)

out = model_medium(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 9.0997e-01, -2.2654e-01,  8.7012e-01,  ...,  1.1012e-01,
          -4.3194e-01,  1.4247e-04],
         [ 6.2650e-01, -2.7274e-01, -3.3416e-01,  ..., -2.6365e-01,
          -6.7231e-01, -1.0621e+00],
         [ 1.2219e+00,  4.4104e-01, -3.9549e-01,  ...,  2.4018e-01,
          -1.0324e-01, -1.0172e-01],
         [ 1.7665e-01, -4.9184e-02, -9.8278e-01,  ...,  4.2390e-02,
          -3.1109e-01, -6.6920e-01]],

        [[ 8.0908e-01, -7.0432e-01,  9.8126e-01,  ...,  9.7242e-01,
           3.0965e-01,  4.4213e-01],
         [ 4.4030e-01, -1.9893e-01, -9.7667e-02,  ..., -1.0905e-01,
           2.4529e-01,  6.5219e-02],
         [ 6.1331e-01,  2.7141e-01, -2.1551e-01,  ...,  9.4755e-02,
           2.2353e-01,  4.8155e-01],
         [ 4.0818e-01,  7.4856e-01, -1.2096e-01,  ..., -4.8255e-01,
          -9.1343e-02, -1.0575e-01]]], grad_fn=<UnsafeViewBackward0>)


In [21]:
total_params = sum(p.numel() for p in model_medium.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 406,212,608


In [22]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [23]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [24]:
model_medium.eval() # disable dropout

out = generate_text_simple(
    model=model_medium,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 25079, 40114,  5797, 40538, 43588, 42500]])
Output length: 10


In [25]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am turnover watt agent 1906north Finder


## Pretraining

In [26]:
import sys
import os

# Get the parent directory (root) and add it to path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root_dir)

from gpt.gpt import GPTModel, generate_text_simple_cached
from config.config import GPT_CONFIG_MEDIUM

In [27]:
print(GPT_CONFIG_MEDIUM)

{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 1024, 'n_heads': 16, 'n_layers': 24, 'drop_rate': 0.1, 'qkv_bias': False}


In [28]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

In [29]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_MEDIUM)
model.eval();

In [30]:
start_context="Every effort moves you"
tokenizer=tiktoken.get_encoding("gpt2")

token_ids=generate_text_simple_cached(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_MEDIUM["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves youowicz ATL scenarios deals9999 RET Jedi automobilesintestinalSET


In [31]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [32]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary
print(probas.shape) # Shape: (batch_size, num_tokens, vocab_size)

torch.Size([2, 3, 50257])


In [33]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

Token IDs:
 tensor([[[36128],
         [23599],
         [45608]],

        [[27303],
         [38692],
         [34213]]])


In [34]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Messispecifiedotide


In [35]:
import torch
import numpy
print(torch.__version__)
print(numpy.__version__)


2.4.0+cu121
1.26.4
